### Import Libraries and run utility notebook

In [0]:
import logging
from datetime import datetime
from pyspark.sql.functions import current_timestamp,to_timestamp,trim,lower,when,max,current_date,col, cast, lit

from delta.tables import delta, DeltaTable
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, Row

In [0]:
%run ../utils/loggingNotebook

In [0]:
%run ../utils/params

In [0]:
 %run ../utils/utilsMethods

In [0]:
%run ../utils/adlsConfig

### Create widgets and getting parameters

In [0]:
sourceObjectName = dbutils.widgets.get("sourceObjectName").strip().lower()
sourceSystem = dbutils.widgets.get("sourceSystem").strip().lower()


### Set logging, pipeline failure and auditing variables and parameters

In [0]:
maxSequence = spark.sql(f"""select max(Cast(reverse(split(adbPipelineId, '_'))[0]AS int)) FROM {pipelineAuditTable} WHERE lower(sourceObjectName) = lower('{sourceObjectName}') and lower(sourceSystem) = lower('{sourceSystem}') and stageOrder = {curatedOrderStage}""").collect()[0][0]

if maxSequence == None:
  sequence = "1"
else:
  sequence = str(maxSequence + 1)

In [0]:
#Fetching the path of the notebook
notebookPath = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebookName = notebookPath.split("/")[-1] #Splitting the path to get notebook name

if sourceObjectName != "" and sourceSystem != "":
  if "." in sourceObjectName:
    schema = sourceObjectName.split('.')[0]
    objectName = sourceObjectName.split('.')[1]
    adbpipelineId = f"{sourceSystem}_{schema}_{objectName}_{curatedStage}_{sequence}"
  else:
    schema = None
    objectName = sourceObjectName
    adbpipelineId = f"{sourceSystem}_{objectName}_{curatedStage}_{sequence}"
    
  properties={'custom_dimensions': {'adbPipelineID':f'{adbpipelineId}','notebookName':f'{notebookName}','sourceName':f'{sourceSystem}','tableName':f'{sourceObjectName}'}}
    
else:
  adbpipelineId = f"{sourceSystem}_{sourceObjectName}_{curatedStage}_{sequence}"
  properties={'custom_dimensions': {'adbPipelineID':f'{adbpipelineId}','notebookName':f'{notebookName}','sourceName':f'{sourceSystem}','tableName':f'{sourceObjectName}'}}
  logger.exception("Empty sourceObjectName/sourceSystem",extra=properties)
  raise Exception("Following field(s) is empty - sourceObjectName, sourceSystem.")
    
# Variables for Audit Table
stageStartTs = datetime.now()
stageObjectName = sourceSystem+objectName
stageName = curatedStage.lower().strip()
stageOrder = int(curatedOrderStage)

In [0]:
try:
  timestamps = spark.sql(f"""SELECT srcStartTs, srcEndTs FROM (SELECT srcStartTs, srcEndTs, dense_rank() over (partition by sourceObjectName, sourceSystem order by stageEndTs desc) AS ranks FROM {pipelineAuditTable} WHERE lower(sourceObjectName) = lower("{sourceObjectName}") and lower(sourceSystem) = lower("{sourceSystem}") and StageOrder = {rawOrderStage}) where ranks = 1""").collect()[0].asDict()
  srcStartTs = timestamps['srcStartTs']
  srcEndTs = timestamps['srcEndTs']

except Exception as e:
  logger.exception(f"Could not fetch srcStartTs and srcEndTs for {sourceObjectName} and {sourceSystem}", extra = properties)


### Extract data from config table and define variables

In [0]:
try:
  #Reading config csv file stored in datalake
  logger.info("Reading data from config file",extra=properties)
  dfConfig = spark.read.csv(jobConfigPath, inferSchema = True, header = True)
  #Converting all string type nulls to None type
  dfConfig = dfConfig.select([when(trim(lower(col(c)))=="null",None).otherwise(col(c)).alias(c) for c in dfConfig.columns])
  
  #Filtering data on sourceObjectName and sourceSystem
  config = dfConfig.where((trim(lower(col("sourceObjectName")))== sourceObjectName.lower()) & (trim(lower(col("sourceSystem")))== sourceSystem.lower())).collect()[0].asDict()
  
  logger.info("Creating variables based on config file", extra=properties)
  workloadType = config['workloadType']
  
  activeFlag = config['activeFlag']
  
  curatedObjectName = objectName
  
  dateColumnName = config['dateColumnName']
  
  primaryKey = config['pkColumnName']
  
  parquetPath = config['parquetPath']
  
  curatedPath = config['curatedPath']

except IndexError as e:
  logger.exception(f"No configuration data found for {sourceObjectName} and {sourceSystem}. Error message: {e}", extra=properties)
  
  #Update Audit Table
  updateAuditTable(curatedFailure,str(e))
  raise e

### Active Flag Check

In [0]:
if activeFlag.strip().lower() == 'y':
  logger.info(f"Active Flag is y for {sourceObjectName}. Notebook will proceed.",extra=properties)
else:
  logger.info(f"Active Flag not Y for {sourceObjectName}. Notebook will NOT proceed.",extra=properties)
  dbutils.notebook.exit("Notebook exited since active flag is NOT Y in config table.")

### Defining ADLS Paths

In [0]:
try:
  logger.info("Defining ADLS paths", extra=properties)
  #rawPath is where parquet files are present
  rawPath = f'{baseAdlsPath}{parquetPath}'
  
  #destinationPath is where delta files will stored
  destinationPath = f'{baseAdlsPath}{curatedPath}'
  
except KeyError as e:
#   If a required key is missing from the configuration data, raise an exception
  logger.exception(f"Missing required key in configuration data with error message: {e}",extra=properties )
  #Update Audit Table
  updateAuditTable(curatedFailure,str(e))
  raise KeyError(f"Missing required key in configuration data. Error message: {e}")
    

### Performing Curated Path Validation

In [0]:
#If workload = historical then path validation will skip. If not then path will be validated.
if workloadType.lower().strip() != "historical" and workloadType.lower().strip() != "histincr":
  pathCheck = filesCheck(baseAdlsPath+curatedPath)
  
  if pathCheck == True:
    logger.info("Curated path exists. Notebook will proceed.", extra = properties)
  else:
    logger.exception("Curated path is incorrect. Check path in config table.", extra=properties)
    #Update Audit Table
    updateAuditTable(curatedFailure,"Curated path is incorrect. Check path in config table.")
    raise Exception("Curated path is incorrect. Check path in config table.")

else:
  logger.info("Curated path check will skip since this is first load.",extra = properties)

In [0]:
#SCD2 Type load is possible when both primary key column and date column are populated in confi table
try:
  if primaryKey != None and dateColumnName != None:
    dataIngestionType = 'SCD2'
    logger.info(f"SCD2 load for {sourceObjectName}", extra=properties)
  else: 
    dataIngestionType = 'Full Load'
    logger.info(f"Full Load for {sourceObjectName}", extra=properties)
    
except Exception as e:
  logger.exception("An error occured while checking whether the table is Full Load or SCD2 Load", extra=properties)
  #Update Audit Table
  updateAuditTable(curatedFailure,str(e))
  raise e

### Processing Raw data based on either Full Load or SCD2 Load

In [0]:
if dataIngestionType == 'Full Load':
  try:
    logger.info(f"Checking whether files for {sourceObjectName} are present in raw path or not", extra=properties)
    #Checking if files at raw path are present or not
    if len(dbutils.fs.ls(rawPath)) != 0:
      filesStatus = True
      logger.info(f"Reading {sourceObjectName} files for Full Load", extra=properties)
      dfRawObject = spark.read.format('parquet').load(rawPath,inferSchema = True, header = True)
      #Check if dataframe is empty or not
      dfEmptyStatus = dfRawObject.count() == 0
      if dfEmptyStatus == False:
        logger.info(f"Adding audit columns to the dataframe",extra = properties)
        dfRawObject = dfRawObject.withColumn("srcSystemName", lit(sourceSystem))\
                                 .withColumn("effectiveStartDate", lit(datetime.now()))\
                                 .withColumn("effectiveEndDate", when(col("cdcIndicator")=="D", lit(datetime.now()))\
                                 . when(col("cdcIndicator")=="A", to_timestamp(lit(futureDateTime),"yyyy-MM-dd HH:mm:ss")))\
                                 .withColumnRenamed("cdcIndicator", "dwIsActive")
        
        #Arranging Columns in pattern: sourceColumns+auditColumns
        #auditColumnList is in params notebook
        rawObjectColumnList = dfRawObject.columns
        finalColumnList = [i for i in rawObjectColumnList if i not in auditColumnList]
        finalColumnList.extend(auditColumnList)
        
        dfRawObject = dfRawObject.select(*finalColumnList)
        dfRawObject.createOrReplaceTempView("tmpVRawObject")
      else:
        logger.info(f"Raw data is empty for {sourceObjectName} and {sourceSystem}",extra=properties)
    else:
      filesStatus = False
      logger.info(f"No files present at raw for {sourceObjectName}",extra=properties)
    
  except Exception as e:
    logger.exception(f"Unable to prepare {sourceObjectName} data for ingestion due to error: {e}",extra=properties)
    #Update Audit Table
    updateAuditTable(curatedFailure,str(e))
    raise e
  
else:
  try:
    logger.info(f"Checking whether files for {sourceObjectName} are present in raw path or not", extra=properties)
    #Checking if files at raw path are present or not
    if len(dbutils.fs.ls(rawPath)) != 0:
      filesStatus = True
      logger.info(f"Reading {sourceObjectName} for SCD Type 2 Load", extra=properties)
      #dfRawObject.isEmpty
      dfRawObject = spark.read.format('parquet').load(rawPath,inferSchema = True, header = True)
      #Check if dataframe is empty or not
      dfEmptyStatus = dfRawObject.count() == 0
      if dfEmptyStatus == False:
        logger.info(f"Adding audit columns to the dataframe",extra = properties)
        dfRawObject = dfRawObject.withColumn("srcSystemName", lit(sourceSystem))\
                                 .withColumn("effectiveEndDate", when(col("cdcIndicator")=="D", col(dateColumnName))\
                                 . when(col("cdcIndicator")=="A", to_timestamp(lit(futureDateTime),"yyyy-MM-dd HH:mm:ss")))\
                                 .withColumnRenamed("cdcIndicator", "dwIsActive")\
                                 .withColumn("effectiveStartDate", col(dateColumnName))
      
        
        #Arranging Columns in pattern: sourceColumns+auditColumns
        #auditColumnList is in params notebook
        rawObjectColumnList = dfRawObject.columns
        finalColumnList = [i for i in rawObjectColumnList if i not in auditColumnList]
        finalColumnList.extend(auditColumnList)
        
        dfRawObject = dfRawObject.select(*finalColumnList)
        #This view will be used for filtering duplicate records
        dfRawObject.createOrReplaceTempView("tmpVRawObject")
      else:
        logger.info(f"Raw data is empty for {sourceObjectName} and {sourceSystem}",extra=properties)
    else:
      filesStatus = False
      logger.info(f"No files present at raw for {sourceObjectName}",extra=properties)
  except FileNotFoundException as d:
      updateAuditTable(curatedFailure,str("File Not Found"))
  except Exception as e:
    logger.exception(f"Unable to prepare {sourceObjectName} data for ingestion due to error: {e}",extra=properties)
    #Update Audit Table
    updateAuditTable(curatedFailure,str(e))
    raise e

In [0]:
#Notebook will exit if there is no data at raw path otherwise proceed to next steps
if filesStatus == False:
  logger.exception(f"Notebook did not run since {sourceObjectName} did not have files at raw path.",extra=properties)
  updateAuditTable(curatedFailure,"No files present at raw path")
  dbutils.notebook.exit(f"Notebook did not run since {sourceObjectName} did not have files at raw path.")
elif dfEmptyStatus == True:
  logger.exception(f"Notebook did not run since {sourceObjectName} did not have data at raw path.", extra=properties)
  updateAuditTable(curatedFailure,"Data at raw path is empty")
  dbutils.notebook.exit(f"Notebook did not run since {sourceObjectName} did not have data at raw path.")
else:
  logger.info(f"Raw data for {sourceObjectName} is not empty. Notebook will proceed", extra=properties)

### Checking if Delta Table Exists

In [0]:
try:
    #check if delta table exists in curated layer or not
    deltaTableStatus = deltaTableCheck(destinationPath)
    
    if deltaTableStatus == True:
      logger.info("Creating delta table dataframe using delta location",extra=properties)

      # Creating dataframe of delta table
      dfDeltaTable = spark.sql(f"Select * from {curatedDbName}.{sourceSystem}{curatedObjectName}")

      # Creating delta table columns list
      deltaColumnList = dfDeltaTable.columns
      
      logger.info(f"{curatedObjectName} Delta table successfully read and dataframe created.", extra=properties)

    else:
      #Writing raw data in curated
      logger.info(f"Writing data for {curatedObjectName} from raw to curated for the first time.", extra=properties)
      
      dfRawObject.write.format("delta").option("mergeSchema", "True").mode("overwrite").save(destinationPath)   
      
except Exception as e:
    logger.exception(f"Error occured either while reading existing delta table or while writing to delta: {e}",extra=properties)

    #Update Audit Table
    updateAuditTable(curatedFailure,str(e))
    raise e

In [0]:
if deltaTableStatus == False:
  #Creating Delta Table
  createDeltaTable(curatedDbName,sourceSystem,curatedObjectName,destinationPath,curatedFileType)
  logger.info("Since delta table did not exist, a new delta table has been created with latest data.", extra=properties)
  #Update Audit Table
  updateAuditTable(curatedSuccess,"")
  dbutils.notebook.exit("SCD2 load completed successfully")
else:
  logger.info("Proceeding to SCD2 Logic.", extra=properties)

### Starting SCD 2 Logic

In [0]:
try:
  logger.info("Comparing raw data with curated to filter duplicate records.", extra=properties)
  if deltaTableStatus == True:
    deltaTableName = f"{curatedDbName}.{sourceSystem}{curatedObjectName}"
    
    primaryKeys = primaryKey.split(',')
    #Dynamically creating join condition for merging based on the merge keys
    joinCondition = " AND ".join("existing.{} = updated.{}".format(k, k) for k in primaryKeys)
    #Condition for updating dwIsActive = A records
    actRecCondition = "{0} AND updated.dwIsActive = existing.dwIsActive".format(joinCondition)
    #Condition for updating dwIsActive = D records
    delRecCondition = "{0} AND updated.dwIsActive = 'D' AND existing.dwIsActive = 'A'".format(joinCondition)
    
    #Filtering out duplicate records and retaining new/updated records with dwIsActive = A
    dfNewUpdatedRecords = spark.sql(f"""SELECT a.* FROM tmpVRawObject a
                                       LEFT JOIN {deltaTableName} b
                                       ON a.dwMd5CheckSum = b.dwMd5CheckSum
                                       WHERE b.dwMd5CheckSum is NULL and a.dwIsActive = 'A'""")
  
    dfNewUpdatedRecords.createOrReplaceTempView("tmpVNewUpdatedRecords")
  
    curatedDelta = DeltaTable.forPath(spark, destinationPath)
  
    curatedDelta.alias("existing")\
    .merge(
            dfNewUpdatedRecords.alias("updated"),\
            #For a primary key, its latest record in curation will become inactive(N) and its effectiveEndDate will become the effectiveStartDate of the new record
          condition = "{}".format(actRecCondition))\
          .whenMatchedUpdate(
                              set = {
                                      "existing.dwIsActive" : "'N'",
                                      "existing.effectiveEndDate" : "updated.effectiveStartDate"
                                      }).whenNotMatchedInsertAll().execute()
    logger.info("Upsert completed in delta table", extra=properties)
  
    #Check for Updated Records which needs to be appended   
    #Filtering records which have to be appended in Delta Table

    dfAppendRecords = spark.sql(f"""SELECT a.* FROM tmpVNewUpdatedRecords a
                                       LEFT JOIN {deltaTableName} b
                                       ON a.dwMd5CheckSum = b.dwMd5CheckSum
                                       WHERE b.dwMd5CheckSum is NULL and a.dwIsActive = 'A'""")
  
    dfAppendRecords = dfAppendRecords.select(*deltaColumnList)
    
#     Appending the new Active Record
    dfAppendRecords.write.format("delta").mode("append").option("mergeSchema","True").save(destinationPath)
    logger.info("Records appended in delta table.", extra=properties)
    
#     Filtering records with dwIsActive = D

    dfDeletedRecords = spark.sql(f"""SELECT a.* FROM tmpVRawObject a
                                       LEFT JOIN {deltaTableName} b
                                       ON a.dwMd5CheckSum = b.dwMd5CheckSum
                                       WHERE b.dwMd5CheckSum is NULL and a.dwIsActive = 'D'""")
    dfDeletedRecords = dfDeletedRecords.select(*deltaColumnList)
  
    #Merging Records with dwIsActive = D
    curatedDelta = DeltaTable.forPath(spark, destinationPath)
  
    if "isDeleted" in deltaColumnList:
      curatedDelta.alias("existing")\
      .merge(
              dfDeletedRecords.alias("updated"),\
              #Merging records with dwIsActive = D
              condition = "{}".format(delRecCondition))\
              .whenMatchedUpdate(
                                  set = {
                                          "existing.dwIsActive" : "'D'",
                                          "existing.effectiveEndDate" : "updated.effectiveEndDate",
                                          "existing.isDeleted" : "updated.isDeleted",
                                          "existing.dwMd5CheckSum" : "updated.dwMd5CheckSum"
                                          }).whenNotMatchedInsertAll().execute()
    else:
      curatedDelta.alias("existing")\
      .merge(
              dfDeletedRecords.alias("updated"),\
              #Merging records with dwIsActive = D
              condition = "{}".format(delRecCondition))\
              .whenMatchedUpdate(
                                  set = {
                                          "existing.dwIsActive" : "'D'",
                                          "existing.effectiveEndDate" : "updated.effectiveEndDate",
                                          "existing.dwMd5CheckSum" : "updated.dwMd5CheckSum"
                                          }).whenNotMatchedInsertAll().execute()
      
    logger.info("Updated records in delta table with dwIsActive = D", extra=properties)
    logger.info("SCD2 Load Successful.", extra=properties)
    
    #Update Audit Table
    updateAuditTable(curatedSuccess,"")
    
  else:
    logger.info("SCD2 Logic did not run since this was first load for the table. New data has been written in curation layer.", extra=properties)
    
except Exception as e:
  logger.exception(f"Data could not be loaded for {sourceObjectName} in delta due to error: {e}",extra=properties)
  #Update Audit Table
  updateAuditTable(curatedFailure,str(e))
  raise e

In [0]:
dbutils.notebook.exit("Curation Notebook Successful.")